In [1]:
%%capture
!pip install transformers accelerate bitsandbytes

In [2]:
import pandas as pd
import numpy as np

# Tokenization utilities
from transformers import AutoTokenizer
from huggingface_hub import login

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Runtime utilities
import math
import time
from tqdm import tqdm, trange

In [3]:
import warnings
warnings.filterwarnings("ignore", message="The attention mask and the pad token id were not set.*")

In [ ]:
login(token='')

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Enables 4-bit quantization
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

2025-08-21 10:21:57.337661: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755771717.707276      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755771717.812903      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

# Load data

In [ ]:
! gdown 

Downloading...
From: https://drive.google.com/uc?id=10yvQXiaHhPEz3iVopohH1TG3mpKI1cr2
To: /kaggle/working/integrated.json
100%|███████████████████████████████████████| 1.55M/1.55M [00:00<00:00, 144MB/s]


In [7]:
import json

with open('/kaggle/working/integrated.json', 'r') as f:
  all_data = json.loads(f.read())
  f.close()

In [8]:
def format_message(input_text):
  messages = [
      {
          "role": "system",
          "content": (
                        """You are an expert annotator of diabetes-related stigma in language.
                        "Your task is to classify the presence and type of stigma in a given input text (a quote, part of an interview, or a social media post).

                        Definition of stigma: any form of discrimination, exclusion, negative stereotypes,
                        shame, blame, or harmful beliefs directed at someone because of diabetes.

                        "Classification steps:\n"
                        "1. Determine if the post expresses stigma:\n"
                        "   - stigma: Contains stigmatizing content or implications.\n"
                        "   - no stigma: Neutral, supportive, or stigma-free.\n\n"
                        "2. If 'stigma', identify all applicable types:\n"
                        "   - Experienced: Direct discrimination or exclusion due to diabetes.\n"
                        "   - Perceived: Awareness or assumption that others hold negative views.\n"
                        "   - Anticipated: Fear or expectation of being stigmatized.\n"
                        "   - Internalized: Shame or self-blame from negative diabetes stereotypes.\n"
                        "   - Intersectional: Stigma compounded by other factors (e.g., obesity, race).\n\n"
                        "   If 'no stigma', return an empty list for Type.\n\n"
                        "3. Provide a brief explanation (max 50 words) for your decision.\n\n"
                        "Respond in this exact format:\n"
                        "Label: [Yes stigma / No stigma]\n"
                        "Type: [List of applicable types or []]\n"
                        "Reasoning: [Your explanation]"

                        Examples:
                        Input: "My fiancé’s mother told him not to marry me because I have diabetes."
                        Output:
                        Label: stigma
                        Type: Experienced
                        Reasoning: This statement shows discrimination and exclusion because the person is being judged as a burden due to diabetes.

                        Input: "I check my blood sugar every morning before breakfast."
                        Output:
                        Label: no stigma
                        Type: []
                        Reasoning: This is a neutral statement describing a health routine, without stigma or negative judgment.

                        Input: "People say diabetes is only for lazy people who eat too much."
                        Output:
                        Label: stigma
                        Type: Perceived
                        Reasoning: The text conveys a negative stereotype linking diabetes to laziness and overeating.
                        """
                    )
      },
      {
          "role": "user",
          "content": f"Text: {input_text}\n Analyze the text for diabetes-related stigma."
      }
  ]

  return messages

# Main loop

In [9]:
tokenizer.pad_token = tokenizer.eos_token

In [10]:
all_responses = []

for data_obj in tqdm(all_data):
    tokenized_chat = tokenizer.apply_chat_template(format_message(data_obj['input text']), tokenize=True, truncation=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    outputs = model.generate(tokenized_chat, max_new_tokens=50, return_dict_in_generate=True, output_scores=True)
    
    response_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=False)
    final_ans = response_text.split('<|eot_id|><|start_header_id|>assistant<|end_header_id|>')[1].strip()

    scores = outputs.scores
    logprobs = [torch.softmax(s, dim=-1).cpu().detach() for s in scores]
    probs = [l.max().item() for l in logprobs]

    res_obj = {'answer': final_ans, 'probs': probs}
    all_responses.append(res_obj)

    if len(all_responses) % 100 == 0:
        with open('Llama_wtype_answers.json', 'w') as f:
            f.write(json.dumps(all_responses))
            f.close()
    

  0%|          | 0/1386 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
  0%|          | 1/1386 [00:06<2:29:26,  6.47s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  0%|          | 2/1386 [00:12<2:18:06,  5.99s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your i

In [11]:
with open('final_Llama_wtype_answers.json', 'w') as f:
    f.write(json.dumps(all_responses))
    f.close()